In [1]:
import pandas as pd
import duckdb

print("pandas version:", pd.__version__)
print("DuckDB version:", duckdb.__version__)
print("Environment setup successful!")

pandas version: 3.0.5
DuckDB version: 1.5.5
Environment setup successful!


In [2]:
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

raw_dir = project_root / "data" / "raw"
raw_files = list(raw_dir.glob("*"))

print("Current directory:", current_dir)
print("Project root:", project_root)
print("Raw data folder:", raw_dir)
print("Files found:")

for file in raw_files:
    print("-", file.name)

Current directory: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\notebooks
Project root: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection
Raw data folder: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\raw
Files found:
- Environmental_Health_Restaurant_and_Market_Inspections.csv


In [4]:
rraw_file = raw_files[0]

df = pd.read_csv(
    raw_file,
    encoding="cp1252",
    low_memory=False
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Rows: 101244
Columns: 20


,ACTIVITY DATE,OWNER ID,OWNER NAME,FACILITY ID,FACILITY NAME,RECORD ID,PROGRAM NAME,PROGRAM STATUS,PROGRAM ELEMENT,PE DESCRIPTION,FACILITY ADDRESS,FACILITY CITY,FACILITY STATE,FACILITY ZIP,SERVICE CODE,SERVICE DESCRIPTION,SCORE,GRADE,SERIAL NUMBER,EMPLOYEE ID
0,07/01/2023,OW0005549,BW BEVERAGE CORP,FA0069428,B W HOTEL LLC,PR0009845,B. W. HOTEL LLC BAKERY,ACTIVE,1631,RESTAURANT (0-30) SEATS MODERATE RISK,9500 WILSHIRE BLVD,BEVERLY HILLS,CA,90212,1,ROUTINE INSPECTION,98,A,DA7CZYLU1,EE0000549
1,07/01/2023,OW0005549,BW BEVERAGE CORP,FA0069428,B W HOTEL LLC,PR0038991,B. W. HOTEL LLC THE BLVD,ACTIVE,1638,RESTAURANT (61-150) SEATS HIGH RISK,9500 WILSHIRE BLVD,BEVERLY HILLS,CA,90212,1,ROUTINE INSPECTION,93,A,DAEBMH1GN,EE0000549
2,07/01/2023,OW0005549,BW BEVERAGE CORP,FA0069428,B W HOTEL LLC,PR0018103,B. W. HOTEL LLC POOL BAR,ACTIVE,1632,RESTAURANT (0-30) SEATS HIGH RISK,9500 WILSHIRE BLVD,BEVERLY HILLS,CA,90212,1,ROUTINE INSPECTION,99,A,DA09SRTPG,EE0000549
3,07/01/2023,OW0121804,LEVY PREMIUM FOOD SERVICE LP,FA0156500,LA CONVENTION CENTER,PR0143275,LA CONVENTION CENTER GALAXY RESTAURANT - LACC,ACTIVE,1641,RESTAURANT (151 + ) SEATS HIGH RISK,1201 S FIGUEROA ST,LOS ANGELES,CA,90015,1,ROUTINE INSPECTION,93,A,DA0EANODJ,EE0000633
4,07/01/2023,OW0121804,LEVY PREMIUM FOOD SERVICE LP,FA0156500,LA CONVENTION CENTER,PR0151559,LA CONVENTION CENTER COMPASS CAFE,ACTIVE,1641,RESTAURANT (151 + ) SEATS HIGH RISK,1201 S FIGUEROA ST,LOS ANGELES,CA,90015,1,ROUTINE INSPECTION,95,A,DALNSQBWJ,EE0000838


In [5]:
raw_df = df.copy()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

print("Shape:", df.shape)
print("\nColumn names:")

for column in df.columns:
    print("-", column)

Shape: (101244, 20)

Column names:
- activity_date
- owner_id
- owner_name
- facility_id
- facility_name
- record_id
- program_name
- program_status
- program_element
- pe_description
- facility_address
- facility_city
- facility_state
- facility_zip
- service_code
- service_description
- score
- grade
- serial_number
- employee_id


In [6]:
grade_text = df["grade"].astype("string").str.strip()

invalid_grade_mask = (
    grade_text.notna()
    & ~grade_text.isin(["A", "B", "C"])
)

invalid_grade_rows = df.loc[
    invalid_grade_mask,
    [
        "activity_date",
        "facility_name",
        "record_id",
        "program_name",
        "program_status",
        "pe_description",
        "service_description",
        "score",
        "grade",
        "serial_number",
        "employee_id"
    ]
]

print("Invalid grade rows:", len(invalid_grade_rows))
display(invalid_grade_rows)

Invalid grade rows: 0


,activity_date,facility_name,record_id,program_name,program_status,pe_description,service_description,score,grade,serial_number,employee_id


In [7]:
# Remove leading and trailing spaces from text columns
text_columns = df.select_dtypes(include=["object", "string"]).columns

df[text_columns] = df[text_columns].apply(
    lambda column: column.astype("string").str.strip()
)

# Convert date and score to appropriate data types
df["activity_date"] = pd.to_datetime(
    df["activity_date"],
    format="%m/%d/%Y",
    errors="coerce"
)

df["score"] = pd.to_numeric(
    df["score"],
    errors="coerce"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing serial numbers:", df["serial_number"].isna().sum())
print("Invalid dates:", df["activity_date"].isna().sum())
print("Missing or invalid scores:", df["score"].isna().sum())

display(
    df["grade"]
      .value_counts(dropna=False)
      .rename_axis("grade")
      .reset_index(name="count")
)

Rows: 101244
Columns: 20
Missing serial numbers: 0
Invalid dates: 0
Missing or invalid scores: 0


,grade,count
0,A,95659
1,B,4547
2,C,935
3,<NA>,103


In [8]:
# Examine key categorical fields

for column in [
    "program_status",
    "service_description",
    "pe_description"
]:
    print(f"\n--- {column} ---")
    display(
        df[column]
          .value_counts(dropna=False)
          .rename_axis(column)
          .reset_index(name="count")
          .head(40)
    )


--- program_status ---


,program_status,count
0,ACTIVE,90404
1,INACTIVE,10746
2,<NA>,94



--- service_description ---


,service_description,count
0,ROUTINE INSPECTION,98998
1,OWNER INITIATED INSPECTION,2246



--- pe_description ---


,pe_description,count
0,RESTAURANT (0-30) SEATS HIGH RISK,19136
1,RESTAURANT (0-30) SEATS MODERATE RISK,17465
2,"FOOD MKT RETAIL (1-1,999 SF) LOW RISK",13078
3,RESTAURANT (31-60) SEATS HIGH RISK,12831
4,RESTAURANT (61-150) SEATS HIGH RISK,10408
5,"FOOD MKT RETAIL (2,000+ SF) LOW RISK",5595
6,RESTAURANT (0-30) SEATS LOW RISK,4621
7,RESTAURANT (151 + ) SEATS HIGH RISK,4542
8,"FOOD MKT RETAIL (1-1,999 SF) HIGH RISK",3496
9,RESTAURANT (31-60) SEATS MODERATE RISK,2945


In [9]:
# Identify restaurant inspection records
restaurant_mask = df["pe_description"].str.startswith(
    "RESTAURANT",
    na=False
)

# Identify currently active restaurant records
active_mask = df["program_status"].eq("ACTIVE")
active_restaurant_mask = restaurant_mask & active_mask

restaurant_df = df.loc[restaurant_mask].copy()
active_restaurant_df = df.loc[active_restaurant_mask].copy()

print("All restaurant inspection records:",
      len(restaurant_df))

print("Active restaurant inspection records:",
      len(active_restaurant_df))

print("Unique active restaurant programs:",
      active_restaurant_df["record_id"].nunique())

print("Unique active restaurant facilities:",
      active_restaurant_df["facility_id"].nunique())

print("Unique active restaurant owners:",
      active_restaurant_df["owner_id"].nunique())

All restaurant inspection records: 76458
Active restaurant inspection records: 67681
Unique active restaurant programs: 29494
Unique active restaurant facilities: 27438
Unique active restaurant owners: 22469


In [10]:
# Keep active restaurant records only
active_restaurants = df.loc[active_restaurant_mask].copy()

# Save an intermediate cleaned dataset
interim_file = (
    project_root
    / "data"
    / "interim"
    / "active_restaurants.csv"
)

active_restaurants.to_csv(
    interim_file,
    index=False,
    encoding="utf-8-sig"
)

print("File saved to:")
print(interim_file)
print("Rows saved:", len(active_restaurants))

File saved to:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\active_restaurants.csv
Rows saved: 67681


In [11]:
# Validate the saved intermediate file
check_df = pd.read_csv(
    interim_file,
    encoding="utf-8-sig",
    low_memory=False
)

print("Rows read back:", len(check_df))
print("Columns read back:", len(check_df.columns))
print("Unique serial numbers:", check_df["serial_number"].nunique())
print("Unique programs:", check_df["record_id"].nunique())
print("Unique facilities:", check_df["facility_id"].nunique())
print("Unique owners:", check_df["owner_id"].nunique())

Rows read back: 67681
Columns read back: 20
Unique serial numbers: 67681
Unique programs: 29494
Unique facilities: 27438
Unique owners: 22469


In [12]:
# Basic profile of active restaurant inspection records

print("Earliest inspection date:", active_restaurants["activity_date"].min())
print("Latest inspection date:", active_restaurants["activity_date"].max())

print(
    "Average score:",
    round(active_restaurants["score"].mean(), 2)
)

print(
    "Median score:",
    active_restaurants["score"].median()
)

print(
    "Minimum score:",
    active_restaurants["score"].min()
)

print(
    "Maximum score:",
    active_restaurants["score"].max()
)

Earliest inspection date: 2023-07-01 00:00:00
Latest inspection date: 2026-06-30 00:00:00
Average score: 94.08
Median score: 95.0
Minimum score: 52
Maximum score: 100


In [13]:
# Keep routine inspections only
routine_active = active_restaurants.loc[
    active_restaurants["service_description"]
    == "ROUTINE INSPECTION"
].copy()

# Sort chronologically
routine_active = routine_active.sort_values(
    ["record_id", "activity_date", "serial_number"]
)

# Keep the latest routine inspection for each permitted program
latest_program_inspections = (
    routine_active
    .drop_duplicates(subset="record_id", keep="last")
    .copy()
)

print("Active restaurant inspection records:", len(active_restaurants))
print("Routine inspection records:", len(routine_active))
print("Programs with a routine inspection:", len(latest_program_inspections))
print(
    "Unique RECORD IDs:",
    latest_program_inspections["record_id"].nunique()
)

display(
    latest_program_inspections[
        [
            "activity_date",
            "record_id",
            "facility_id",
            "facility_name",
            "score",
            "grade"
        ]
    ].head()
)

Active restaurant inspection records: 67681
Routine inspection records: 65810
Programs with a routine inspection: 29494
Unique RECORD IDs: 29494


,activity_date,record_id,facility_id,facility_name,score,grade
64848,2025-04-07,PR0000004,FA0150765,TERRANEA,98,A
81339,2025-10-17,PR0000005,FA0059741,CHIPOTLE MEXICAN GRILL,95,A
61298,2025-03-06,PR0000012,FA0023865,MARMALADE CAFE RESTAURANT,96,A
99498,2026-06-03,PR0000021,FA0013174,CARL'S JR,96,A
60537,2025-02-28,PR0000023,FA0044138,SOUTHERN CALIFORNIA PIZZA CO LLC,96,A


In [14]:
# Count permitted programs at each physical facility
facility_program_counts = (
    latest_program_inspections
    .groupby("facility_id")["record_id"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Unique physical facilities:",
    latest_program_inspections["facility_id"].nunique()
)

print(
    "Facilities with more than one program:",
    (facility_program_counts > 1).sum()
)

print(
    "Maximum programs at one facility:",
    facility_program_counts.max()
)

display(
    facility_program_counts
      .head(10)
      .rename("program_count")
      .reset_index()
)

Unique physical facilities: 27438
Facilities with more than one program: 902
Maximum programs at one facility: 121


,facility_id,program_count
0,FA0289131,121
1,FA0019271,73
2,FA0332645,48
3,FA0006427,43
4,FA0065100,35
5,FA0153604,35
6,FA0170909,33
7,FA0275668,31
8,FA0170678,27
9,FA0154994,24


In [15]:
largest_facility_id = "FA0289131"

largest_facility_records = (
    latest_program_inspections.loc[
        latest_program_inspections["facility_id"] == largest_facility_id,
        [
            "facility_id",
            "facility_name",
            "facility_address",
            "facility_city",
            "record_id",
            "program_name",
            "pe_description",
            "score"
        ]
    ]
    .sort_values(["facility_name", "record_id"])
)

print("Rows:", len(largest_facility_records))
print(
    "Unique names:",
    largest_facility_records["facility_name"].nunique()
)
print(
    "Unique addresses:",
    largest_facility_records["facility_address"].nunique()
)
print(
    "Unique cities:",
    largest_facility_records["facility_city"].nunique()
)

display(largest_facility_records.head(20))

Rows: 121
Unique names: 1
Unique addresses: 1
Unique cities: 1


,facility_id,facility_name,facility_address,facility_city,record_id,program_name,pe_description,score
91097,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244562,LEVEL 2 - EAST OWNERS CLUB KITCHEN,RESTAURANT (151 + ) SEATS HIGH RISK,99
91142,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244563,LEVEL 2 - WEST OWNERS CLUB KITCHEN,RESTAURANT (151 + ) SEATS HIGH RISK,99
73392,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244564,LEVEL 2 - EAST OWNERS SUITE PANTRY,RESTAURANT (0-30) SEATS LOW RISK,98
80104,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244565,LEVEL 2 - WEST OWNERS SUITE PANTRY- 50 YRD LINE,RESTAURANT (0-30) SEATS LOW RISK,99
85467,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244790,LEVEL 4 - 232 OLVERA TACOS & MARKET,RESTAURANT (151 + ) SEATS MODERATE RISK,99
91141,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244791,LEVEL 4 - 230 SAWTELLE BBQ,RESTAURANT (151 + ) SEATS HIGH RISK,98
85462,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244792,LEVEL 4 - 235 FAIRFAX,RESTAURANT (151 + ) SEATS MODERATE RISK,99
85476,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244793,LEVEL 4 - 234 POPCORN ROOM & KEG PANTRY,RESTAURANT (0-30) SEATS MODERATE RISK,99
91090,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244794,LEVEL 6 - 340 FAIRFAX BURGER / SAN VICENTE PIZZA,RESTAURANT (151 + ) SEATS MODERATE RISK,98
91091,FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,PR0244795,LEVEL 6 - 333 SAWTELLE CHICKEN / OLVERA TACOS,RESTAURANT (151 + ) SEATS MODERATE RISK,96


In [17]:
# Examine the distribution of programs per physical facility

program_count_distribution = pd.cut(
    facility_program_counts,
    bins=[0, 1, 2, 3, 5, 10, float("inf")],
    labels=["1", "2", "3", "4–5", "6–10", "11+"]
).value_counts().sort_index()

display(
    program_count_distribution
      .rename_axis("programs_per_facility")
      .reset_index(name="facility_count")
)

print(
    "Facilities with more than 10 programs:",
    (facility_program_counts > 10).sum()
)

,programs_per_facility,facility_count
0,1,26536
1,2,543
2,3,231
3,4–5,73
4,6–10,37
5,11+,18


Facilities with more than 10 programs: 18


In [18]:
# Inspect all facilities with more than 10 programs

large_facility_ids = facility_program_counts[
    facility_program_counts > 10
].index

large_multi_program_facilities = (
    latest_program_inspections.loc[
        latest_program_inspections["facility_id"].isin(large_facility_ids)
    ]
    .groupby("facility_id")
    .agg(
        facility_name=("facility_name", "first"),
        facility_address=("facility_address", "first"),
        facility_city=("facility_city", "first"),
        program_count=("record_id", "nunique"),
        owner_count=("owner_id", "nunique")
    )
    .sort_values("program_count", ascending=False)
)

print("Large multi-program facilities:", len(large_multi_program_facilities))
display(large_multi_program_facilities)

Large multi-program facilities: 18


,facility_name,facility_address,facility_city,program_count,owner_count
facility_id,,,,,
FA0289131,SOFI STADIUM,1000 S PRAIRIE AVE,INGLEWOOD,121,1
FA0019271,DODGERS STADIUM,1000 VIN SCULLY AVE,LOS ANGELES,73,1
FA0332645,INTUIT DOME,3930 W CENTURY BLVD,INGLEWOOD,48,1
FA0006427,CRYPTOCOM ARENA (LEVY),1111 S FIGUEROA ST #113,LOS ANGELES,43,1
FA0153604,SANTA ANITA PARK,285 W HUNTINGTON DR,ARCADIA,35,1
FA0065100,LA COLISEUM,3911 S FIGUEROA ST,LOS ANGELES,35,1
FA0170909,UNIVERSAL STUDIOS HOLLYWOOD,100 UNIVERSAL CITY PLZ,UNIVERSAL CITY,33,1
FA0275668,LOS ANGELES MEMORIAL COLISEUM,3911 S FIGUEROA ST,LOS ANGELES,31,1
FA0170678,SIX FLAGS MAGIC MOUNTAIN & SIX FLAGS HURRICANE...,26101 MAGIC MOUNTAIN PKWY,VALENCIA,27,1


In [19]:
# Create one row per physical restaurant facility

facility_level = (
    latest_program_inspections
    .groupby("facility_id", as_index=False)
    .agg(
        facility_name=("facility_name", "first"),
        facility_address=("facility_address", "first"),
        facility_city=("facility_city", "first"),
        facility_state=("facility_state", "first"),
        facility_zip=("facility_zip", "first"),
        latest_inspection_date=("activity_date", "max"),
        program_count=("record_id", "nunique"),
        owner_count=("owner_id", "nunique"),
        average_latest_score=("score", "mean"),
        minimum_latest_score=("score", "min"),
        maximum_latest_score=("score", "max")
    )
)

# Mark unusually large multi-program venues
facility_level["special_venue"] = (
    facility_level["program_count"] > 10
)

# Round the aggregated score
facility_level["average_latest_score"] = (
    facility_level["average_latest_score"].round(2)
)

print("Facility-level rows:", len(facility_level))
print(
    "Special venues:",
    facility_level["special_venue"].sum()
)
print(
    "Standard restaurant facilities:",
    (~facility_level["special_venue"]).sum()
)

display(facility_level.head())

Facility-level rows: 27438
Special venues: 18
Standard restaurant facilities: 27420


,facility_id,facility_name,facility_address,facility_city,facility_state,facility_zip,latest_inspection_date,program_count,owner_count,average_latest_score,minimum_latest_score,maximum_latest_score,special_venue
0,FA0001114,CHRIS & PITTS BBQ #6,9243 LAKEWOOD BLVD,DOWNEY,CA,90240,2024-11-13,1,1,90.0,90,90,False
1,FA0001155,FRATERNAL ORDER OF EAGLES,13018 W WASHINGTON BLVD,LOS ANGELES,CA,90066,2026-06-18,1,1,97.0,97,97,False
2,FA0001334,HARBOR ROOM BAR,195 CULVER BLVD,PLAYA DEL REY,CA,90293,2024-07-25,1,1,98.0,98,98,False
3,FA0001348,JACK IN THE BOX,9433 RESEDA BLVD,NORTHRIDGE,CA,91324,2025-04-23,1,1,98.0,98,98,False
4,FA0001371,AVALON SEAFOOD,20 GREEN PLEASURE PIER,AVALON,CA,90704,2025-07-14,1,1,93.0,93,93,False


In [20]:
# Save the facility-level dataset

facility_file = (
    project_root
    / "data"
    / "interim"
    / "active_restaurant_facilities.csv"
)

facility_level.to_csv(
    facility_file,
    index=False,
    encoding="utf-8-sig"
)

print("File saved to:")
print(facility_file)
print("Rows saved:", len(facility_level))

File saved to:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\active_restaurant_facilities.csv
Rows saved: 27438


In [21]:
# Check whether one address contains multiple facility IDs

valid_address_facilities = facility_level.loc[
    facility_level["facility_address"].notna()
    & facility_level["facility_address"].ne("")
].copy()

address_summary = (
    valid_address_facilities
    .groupby(
        ["facility_city", "facility_address"],
        as_index=False
    )
    .agg(
        facility_count=("facility_id", "nunique"),
        facility_names=("facility_name", "nunique"),
        total_programs=("program_count", "sum")
    )
    .sort_values(
        ["facility_count", "total_programs"],
        ascending=False
    )
)

multi_facility_addresses = address_summary.loc[
    address_summary["facility_count"] > 1
].copy()

print(
    "Addresses with multiple facility IDs:",
    len(multi_facility_addresses)
)

print(
    "Maximum facility IDs at one address:",
    multi_facility_addresses["facility_count"].max()
)

display(multi_facility_addresses.head(20))

Addresses with multiple facility IDs: 587
Maximum facility IDs at one address: 12


,facility_city,facility_address,facility_count,facility_names,total_programs
17781,NORTHRIDGE,18111 NORDHOFF ST,12,12,12
13715,LOS ANGELES,5151 STATE UNIVERSITY DR,9,9,9
9068,LOS ANGELES,10250 SANTA MONICA BLVD,8,8,9
18703,PALMDALE,5550 PEARBLOSSOM HWY,8,8,8
2014,BURBANK,2627 N HOLLYWOOD WAY,7,7,8
14869,LOS ANGELES,700 WORLD WAY,7,7,8
14876,LOS ANGELES,7000 HOLLYWOOD BLVD,7,5,8
24270,UNIVERSAL CITY,100 UNIVERSAL CITY PLZ,6,6,40
1357,BELL GARDENS,888 BICYCLE CASINO DR,6,6,14
25812,WESTLAKE VILLAGE,2 DOLE DR,6,6,9
